# 03 · Committee relevance

The salience tests in notebook 02 split Senate purchases by **committee relevance**.
This notebook shows how that label is built and saves the committee map used in the appendix.

A purchase is *committee-relevant* when the senator sat on a committee whose legal
jurisdiction (Senate Rule XXV) covers the stock's industry (its two-digit NAICS sector).
We combine three inputs:

1. **Senate purchases** from Quiver (licensed).
2. **Committee rosters** from the public `unitedstates/congress-legislators` project.
3. **Stock industry codes** (NAICS) from Compustat (licensed).

The committee-to-NAICS map (`senate_committee_naics_map.yaml`) is hand-built from Rule XXV.

In [1]:
import sys
from pathlib import Path

here = Path.cwd()
ROOT = here if (here / "helpers.py").exists() else here.parent
sys.path.insert(0, str(ROOT))

import pandas as pd

from config import TABLE_DIR, LICENSED_DATA
from helpers import (load_quiver, purchase_events, build_committee_match,
                     committee_mapping_table)

## Build the match

We take the Senate purchases, look up each senator's committees in the Congress that was
sitting on the filing day, and check whether any of those committees cover the stock's sector.
The result is one row per ticker and filing day.

In [2]:
# Trading days, read cheaply from just the date column of the panel.
days = pd.Index(sorted(pd.read_parquet(LICENSED_DATA / "work_panel.parquet", columns=["date"])["date"].unique()))

quiver = load_quiver()
senate_rows = purchase_events(quiver, "senate", days)
match = build_committee_match(senate_rows)

# Sanity check: does the rebuild agree with the cached file notebook 02 used?
cached = pd.read_parquet(LICENSED_DATA / "ticker_day_match_senate.parquet")
check = match.merge(cached[["ticker", "filed_date", "committee_match"]],
                    on=["ticker", "filed_date"], how="inner", suffixes=("", "_cached"))
agree = (check["committee_match"].fillna(-9) == check["committee_match_cached"].fillna(-9)).mean()
print(f"Rebuilt rows: {len(match)}; overlap with cached: {len(check)}; agreement: {agree:.1%}")

match.to_parquet(LICENSED_DATA / "ticker_day_match_senate.parquet", index=False)
print("Ticker-day rows:", len(match))
match.head()


Rebuilt rows: 1421; overlap with cached: 1421; agreement: 100.0%
Ticker-day rows: 1421


,ticker,filed_date,committee_match,naics_2,n_trades
0,0QZI.IL,2019-10-21,<NA>,None,2
1,3V64.TI,2019-10-21,<NA>,None,2
2,A,2023-05-11,1,33,1
3,A,2024-05-15,1,33,1
4,AA,2021-07-23,1,33,4


## Classification breakdown

Most purchases are **not** committee-relevant. Many cannot be classified at all, usually
because the senator's committee map has no clear industry link.

In [3]:
labels = {1: "committee-relevant", 0: "not relevant"}
breakdown = match["committee_match"].map(labels).fillna("unclassified").value_counts()
breakdown = breakdown.rename_axis("class").reset_index(name="ticker_days")
breakdown["share"] = (breakdown["ticker_days"] / len(match)).round(3)
breakdown

,class,ticker_days,share
0,not relevant,1033,0.727
1,committee-relevant,327,0.230
2,unclassified,61,0.043


## Committee-to-NAICS map

The full mapping for both chambers. House rows follow Dong and Xu (2025); Senate rows are
author-coded from Rule XXV. `Not mapped` committees are procedural and create no match.

In [4]:
mapping = committee_mapping_table()
mapping.to_csv(TABLE_DIR / "committee_naics_mapping.csv", index=False)
print("Committees mapped:", len(mapping))
mapping[mapping["chamber"] == "Senate"]

Committees mapped: 44


,chamber,committee,naics_2_codes
23,Senate,"Agriculture, Nutrition, and Forestry",11
24,Senate,Appropriations,Not mapped
25,Senate,Armed Services,"33, 54"
26,Senate,"Banking, Housing, and Urban Affairs","52, 53"
27,Senate,Budget,Not mapped
28,Senate,"Commerce, Science, and Transportation","48, 49, 51, 54, 71"
29,Senate,Energy and Natural Resources,"21, 22"
30,Senate,Environment and Public Works,"23, 22, 56"
31,Senate,Finance,52
32,Senate,Foreign Relations,"92, 54"


The price effect by committee relevance is in notebook 02 (the salience table). There the
committee-relevant group shows a **smaller** return effect than the rest, which is the
opposite of what a committee-information story would predict.